In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import scikit_posthocs as sp
import scipy.stats as stats

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'font.size': 12, 'figure.figsize': (10, 5)})

power_modes = ['07-watt', 'MAXN_SUPER']
power_labels = {'07-watt': '7W Constrained', 'MAXN_SUPER': 'MAXN_SUPER'}

languages = ['cpp', 'rust', 'python']
lang_colours = {'cpp': '#004482', 'rust': '#D34516', 'python': '#82B043'}
lang_labels = {'cpp': "C++20", 'rust': "Rust 1.97.1", 'python': "CPython 3.10.12"}

policies = ['BoundedQueue', 'ExponentialBackoff', 'DropOldest', 'DropNewest', 'AdaptiveDecimation']
policy_labels = ["Bounded\nQueue", "Exponential\nBackoff", "Drop\nOldest", "Drop\nNewest", "Adaptive\nDecimation"]
policy_colours = {
    'BoundedQueue': '#4B5563',
    'ExponentialBackoff': '#6B21A8',
    'DropOldest': '#15803D',
    'DropNewest': '#BE123C',
    'AdaptiveDecimation': '#854D0E'
}

loads = ['0.01', '0.02', '0.03', '0.04', '0.05', '0.06', '0.07',
         '1.0', '1.5', '2.0', '2.5',
         '4.0', '5.5', '7.0', '8.5', '10.0', '12.5', '15.0', '17.5', '20.0']

streams = ['RGB', 'Accelerometer', 'Gyroscope']

marker = '.'
markersize = 5
markeralpha = 1.0
linestyle = '--'
linewidth = 1
linecol = 'red'

Path('./img/').mkdir(exist_ok=True)
img_extension = 'pdf'
img_dpi = 300

In [ ]:
MIN_LOAD = 0.00

for mode in power_modes:
    max_loads = {p: {l: MIN_LOAD for l in languages} for p in policies}

    for policy in policies:
        for lang in languages:
            highest_load = MIN_LOAD

            for load in loads:
                if not os.path.isdir(f'{mode}/{lang}/{policy}/{load}/'):
                    continue

                is_saturated = False

                for stream in streams:
                    # Read the telemetery and discard the first ten seconds to account for cold-start
                    df = pd.read_csv(f'{mode}/{lang}/{policy}/{load}/telemetry_{stream}.csv').iloc[10:]

                    # Total loss is the sum of the dropped frames and lapped frames
                    total_loss = df['dropped_frames'].sum() + df['lapped_frames'].sum()

                    if total_loss > 0:
                        is_saturated = True
                        break

                if is_saturated:
                    break

                highest_load = float(load)

            max_loads[policy][lang] = highest_load

    width = 0.25
    x = np.arange(len(policies))
    fig, ax = plt.subplots(figsize=(10, 5))

    for i, lang in enumerate(languages):
        curr_loads = [max_loads[p][lang] for p in policies]
        offset = (i - (len(languages) - 1) / 2) * width
        bars = ax.bar(x + offset, curr_loads, width, label=lang_labels[lang], color=lang_colours[lang], alpha=0.8)
        labels = ["< 0.01" if v == MIN_LOAD else str(v) for v in curr_loads]
        ax.bar_label(bars, labels=labels, padding=3, fontsize=10)

    ax.set_ylim(0, 6.0)
    ax.set_axisbelow(True)
    ax.grid(axis='x', visible=False)
    ax.grid(axis='y', linestyle='--', alpha=1.0)

    ax.set_ylabel("Maximum Sustained Load Multiplier")
    ax.set_xticks(x)
    ax.set_xticklabels(policy_labels)

    ax.legend(frameon=True, facecolor='white', edgecolor='gray', framealpha=1.0)

    fig.tight_layout()
    plt.savefig(f'img/{mode}-baseline-performance.{img_extension}', dpi=img_dpi)
    plt.show()

In [ ]:
DEADLINE_NS = 100_000_000
848_822_271

for policy in policies:
    print(f"\n{policy}")
    for lang in languages:
        highest_load = 0.0

        for load in loads:
            dir_path = f'{mode}/{lang}/{policy}/{load}/'
            if not os.path.isdir(dir_path):
                continue

            df = pd.read_csv(f'{dir_path}/telemetry_RGB.csv').iloc[10:]

            if (df['total_latency_max'] > DEADLINE_NS).any():
                break
                
            highest_load = float(load)
            
        print(f"  {lang:<8}: {highest_load}")

In [ ]:
def total_latency(languages, load):
    plt.figure(figsize=(10, 5))

    for lang in languages:
        df = pd.read_csv(f'MAXN_SUPER/{lang}/ExponentialBackoff/{load}/telemetry_RGB.csv').iloc[10:]
        df = df['total_latency_p99_9'] / 1_000_000.0
        df = df.sort_values()
        p = 1. * np.arange(len(df)) / (len(df) - 1)

        plt.plot(df, p, marker=marker, markersize=markersize, linestyle='none', 
            color=lang_colours[lang], alpha=markeralpha, label=lang_labels[lang])

        pct = (len(df[df > 100.0]) / len(df)) * 100
        print(f"{lang}: {pct:.1f}% breached the deadline with a range of [{df.min():.1f}, {df.max():.1f}] ms.")

    plt.axvline(x=100.0, color=linecol, linestyle=linestyle, linewidth=linewidth, label="100ms Deadline")
    plt.xlabel("Latency (ms)")
    plt.ylabel("Cumulative Probability")

    plt.xscale('log')
    plt.grid(linestyle='--', alpha=1.0)
    plt.legend(loc='upper left', frameon=True, facecolor='white', edgecolor='gray', framealpha=1.0, markerscale=2.0)
    plt.tight_layout()
    plt.savefig(f'img/MAXN_SUPER-latency-{load}.{img_extension}', dpi=img_dpi)
    plt.show()

total_latency(languages, '5.5')
total_latency(['cpp', 'rust'], '7.0')

In [ ]:
plt.figure(figsize=(10, 5))

def plot_cdf(mode, lang, color, linestyle='-'):
    df = pd.read_csv(f'{mode}/{lang}/ExponentialBackoff/5.5/telemetry_RGB.csv').iloc[10:]
    df = df['total_latency_p99_9'] / 1_000_000.0
    df = df.sort_values()
    p = 1. * np.arange(len(df)) / (len(df) - 1)
    plt.plot(df, p, marker='.', linestyle='none', color=color, alpha=0.6, label=f"{lang_labels[lang]} ({power_labels[mode]})")

    pct = (len(df[df > 100.0]) / len(df)) * 100
    print(f"{lang} {mode}: {pct:.1f}% breached the deadline with a maximum of {df.max():.1f} ms.")

plot_cdf('MAXN_SUPER', 'cpp', 'blue')
plot_cdf('MAXN_SUPER', 'rust', 'cyan')
plot_cdf('07-watt', 'cpp', 'red')
plot_cdf('07-watt', 'rust', 'orange')

plt.axvline(x=100.0, color=linecol, linestyle=linestyle, linewidth=linewidth, label="100ms Deadline")
plt.xscale('log')
plt.grid(linestyle='--', alpha=1.0)
plt.xlabel("Latency (ms)")
plt.ylabel("Cumulative Probability")
plt.legend(frameon=True, facecolor='white', edgecolor='gray', framealpha=1.0, markerscale=2.0)
plt.tight_layout()
plt.savefig('img/latency-comparison.pdf', dpi=img_dpi)
plt.show()

In [ ]:
MIN_LOAD = 0.00
policy = 'ExponentialBackoff'
mode = 'MAXN_SUPER'
comp_langs = ['cpp', 'rust']

max_loads = {lang: {stream: MIN_LOAD for stream in streams} for lang in comp_langs}

for lang in comp_langs:
    for stream in streams:
        highest_load = MIN_LOAD

        for load in loads:
            if not os.path.isdir(f'{mode}/{lang}/{policy}/{load}/'):
                continue

            df = pd.read_csv(f'{mode}/{lang}/{policy}/{load}/telemetry_{stream}.csv').iloc[10:]
            total_loss = df['dropped_frames'].sum() + df['lapped_frames'].sum()

            if total_loss > 0:
                break

            highest_load = float(load)

        max_loads[lang][stream] = highest_load

width = 0.35
x = np.arange(len(streams))
fig, ax = plt.subplots(figsize=(10, 5))

for i, lang in enumerate(comp_langs):
    curr_loads = [max_loads[lang][stream] for stream in streams]
    offset = (i - (len(comp_langs) - 1) / 2) * width
    bars = ax.bar(x + offset, curr_loads, width, label=lang_labels[lang], color=lang_colours[lang], alpha=0.8)
    labels = ["< 0.01" if v == MIN_LOAD else str(v) for v in curr_loads]
    ax.bar_label(bars, labels=labels, padding=3, fontsize=10)

ax.set_axisbelow(True)
ax.grid(axis='x', visible=False)
ax.grid(axis='y', linestyle='--', alpha=1.0)

ax.set_ylabel("Maximum Sustained Load Multiplier")
ax.set_xticks(x)
ax.set_xticklabels(streams)

ax.legend(frameon=True, facecolor='white', edgecolor='gray', framealpha=1.0)

fig.tight_layout()
plt.savefig(f'img/{mode}-stream-saturation.{img_extension}', dpi=img_dpi)
plt.show()

In [ ]:
stages = [
    'unbounded_wait_p50',
    'idiomatic_wait_p50',
    'inference_exec_p50',
    'mpsc_wait_p50',
    'fusion_exec_p50'
]

stage_labels = [
    'Unbounded Queue Wait', 
    'Bounded Queue Wait', 
    'Inference Execution', 
    'MPSC Wait', 
    'Fusion Execution'
]

means = {stage: [] for stage in stages}

for lang in languages:
    df = pd.read_csv(f'MAXN_SUPER/{lang}/ExponentialBackoff/0.04/telemetry_RGB.csv').iloc[10:]
    total = 0.0

    for stage in stages:
        m = df[stage].mean() / 1_000_000.0
        means[stage].append(m)
        print(f"{lang} {stage}: {m} ms")
        total += m

    print(f"{lang} total: {total} ms")

x = np.arange(len(languages))
width = 0.5
bottoms = np.zeros(len(languages))
colours = ['#e41a1c', '#ff7f00', '#4daf4a', '#377eb8', '#984ea3']

fig, ax = plt.subplots(figsize=(10, 4))

for i, stage in enumerate(stages):
    ax.bar(x, means[stage], width, label=stage_labels[i], bottom=bottoms, color=colours[i], edgecolor='black')
    bottoms += np.array(means[stage])

ax.set_ylabel("Average Median Latency (ms)")
ax.set_xticks(x)
ax.set_xticklabels(lang_labels.values())
ax.grid(axis='x', visible=False)
ax.grid(axis='y', linestyle='--', alpha=1.0)

plt.axhline(y=100.0, color=linecol, linestyle=linestyle, linewidth=linewidth, label="100ms Deadline")
ax.legend(title="Pipeline Stage", frameon=True, facecolor='white', edgecolor='gray', framealpha=1.0)

fig.tight_layout()
plt.savefig(f'img/MAXN_SUPER-latency-breakdown.pdf', dpi=img_dpi)
plt.show()

In [ ]:
def dropped_frames(lang, load):
    plt.figure(figsize=(10, 5))

    df = pd.read_csv(f'MAXN_SUPER/{lang}/DropOldest/{load}/telemetry_RGB.csv').iloc[10:]
    df['cumulative_dropped'] = df['dropped_frames'].cumsum()
    plt.plot(range(len(df)), df['cumulative_dropped'], label="Drop Oldest", color=policy_colours['DropOldest'], linewidth=2.0)
    print(f"Drop Oldest dropped frames: {df['cumulative_dropped'].max()}")

    df = pd.read_csv(f'MAXN_SUPER/{lang}/DropNewest/{load}/telemetry_RGB.csv').iloc[10:]
    df['cumulative_dropped'] = df['dropped_frames'].cumsum()
    plt.plot(range(len(df)), df['cumulative_dropped'], label="Drop Newest", color=policy_colours['DropNewest'], linewidth=2.0)
    print(f"Drop Newest dropped frames: {df['cumulative_dropped'].max()}")

    df = pd.read_csv(f'MAXN_SUPER/{lang}/AdaptiveDecimation/{load}/telemetry_RGB.csv').iloc[10:]
    df['cumulative_dropped'] = df['dropped_frames'].cumsum()
    # plt.plot(range(len(df)), df['cumulative_dropped'], label="Adaptive Decimation (Dynamic Load-Shedding)", color=policy_colours['AdaptiveDecimation'], linewidth=linewidth)
    print(f"Adaptive Decimation dropped frames: {df['cumulative_dropped'].max()}")

    df = pd.read_csv(f'MAXN_SUPER/{lang}/ExponentialBackoff/{load}/telemetry_RGB.csv').iloc[10:]
    df['cumulative_dropped'] = df['dropped_frames'].cumsum()
    plt.plot(range(len(df)), df['cumulative_dropped'], label="Exponential Backoff", color=policy_colours['ExponentialBackoff'], linewidth=2.0)
    print(f"Flow-Control dropped frames: {df['cumulative_dropped'].max()}")

    plt.xlabel("Time (seconds)")
    plt.ylabel("Total Dropped Frames")
    plt.grid(linestyle='--', alpha=1.0)
    plt.legend(loc='upper left', frameon=True, facecolor='white', edgecolor='gray', framealpha=1.0)
    plt.tight_layout()
    plt.savefig(f'img/MAXN_SUPER-dropped-frames.{img_extension}', dpi=img_dpi)

    return plt

dropped_frames('rust', '2.5').show()

In [ ]:
policies_map = {
    'DropOldest': ("Drop Oldest", 'green'),
    'AdaptiveDecimation': ("Adaptive Decimation", 'orange'),
    'DropNewest': ("Drop Newest", 'tab:purple'),
    'ExponentialBackoff': ("Exponential Backoff", 'blue'),
}

plt.figure(figsize=(10, 6))

for policy, (label, c) in policies_map.items():
    df = pd.read_csv(f'MAXN_SUPER/rust/{policy}/7.0/telemetry_RGB.csv').iloc[10:]
    df = df['total_latency_p99_9'] / 1_000_000.0
    df = df.sort_values()
    p = 1. * np.arange(len(df)) / (len(df) - 1)
    plt.plot(df, p, marker=marker, linestyle='none', color=policy_colours[policy], alpha=markeralpha, label=label)

    pct = (len(df[df > 100.0]) / len(df)) * 100
    print(f"{label}: {pct:.1f}% breached the deadline with a maximum of {df.max():.1f} ms.")

plt.axvline(x=100.0, color=linecol, linestyle=linestyle, linewidth=linewidth, label='100ms Deadline')
plt.xscale('log')
plt.xlabel("Latency (ms)")
plt.ylabel("Cumulative Probability")
plt.grid(linestyle='--', alpha=1.0)
plt.legend(frameon=True, facecolor='white', edgecolor='gray', framealpha=1.0, markerscale=2.0)
plt.tight_layout()
plt.savefig(f'img/MAXN_SUPER-latency-comparison.pdf', dpi=img_dpi)
plt.show()

In [ ]:
time_seq = range(60)

fig, ax1 = plt.subplots(figsize=(10, 4))
ax1.set_xlabel("Time (seconds)")
ax1.set_ylabel("Max Latency (ms)", color='black')

df = pd.read_csv(f'MAXN_SUPER/python/ExponentialBackoff/0.04/telemetry_RGB.csv').head(60)
gc_pause_ms = df['gc_pause_ns'] / 1_000_000.0
df = df['total_latency_max'] / 1_000_000.0
ax1.plot(time_seq, df, color='tab:green', linewidth=2, label=lang_labels['python'])
print(f"Python maximum latencies: startup={df.iloc[:10].max()}, steady={df.iloc[10:].max()}")
print(f"Python variance: startup={df.iloc[:10].max() - df.iloc[:10].min()}, steady={df.iloc[10:].max() - df.iloc[10:].min()}\n")

df = pd.read_csv(f'MAXN_SUPER/cpp/ExponentialBackoff/0.04/telemetry_RGB.csv').head(60)
df = df['total_latency_max'] / 1_000_000.0
ax1.plot(time_seq, df, color='blue', linewidth=2, label=lang_labels['cpp'])
print(f"C++ maximum latencies: startup={df.iloc[:10].max()}, steady={df.iloc[10:].max()}")
print(f"C++ variance: startup={df.iloc[:10].max() - df.iloc[:10].min()}, steady={df.iloc[10:].max() - df.iloc[10:].min()}\n")

df = pd.read_csv(f'MAXN_SUPER/rust/ExponentialBackoff/0.04/telemetry_RGB.csv').head(60)
df = df['total_latency_max'] / 1_000_000.0
ax1.plot(time_seq, df, color='orange', linewidth=2, label=lang_labels['rust'])
print(f"Rust maximum latencies: startup={df.iloc[:10].max()}, steady={df.iloc[10:].max()}")
print(f"Rust variance: startup={df.iloc[:10].max() - df.iloc[:10].min()}, steady={df.iloc[10:].max() - df.iloc[10:].min()}")

ax1.tick_params(axis='y', labelcolor='black')
ax1.axhline(y=100.0, color='red', linestyle='--', alpha=0.5, label="100ms Deadline")
ax1.axvline(x=10.0, color='gray', linestyle=':', linewidth=2, label="Initialisation Boundary")

ax2 = ax1.twinx()  
ax2.set_ylabel("Python GC Pause Duration (ms)", color='tab:purple')
ax2.bar(time_seq, gc_pause_ms, color='tab:purple', alpha=0.4, width=1.0, label="GC Pause")
ax2.tick_params(axis='y', labelcolor='tab:purple')
ax2.set_ylim(0, max(10, gc_pause_ms.max() * 1.2))

# fig.legend(loc='upper right', bbox_to_anchor=(0.935, 0.95), frameon=True, facecolor='white', edgecolor='gray', framealpha=1.0)
fig.legend(bbox_to_anchor=(1.25, 0.95), loc='upper right', borderaxespad=0.0, frameon=True, facecolor='white', edgecolor='gray', framealpha=1.0)
fig.tight_layout()
plt.savefig(f'img/MAXN_SUPER-python-gc.pdf', dpi=img_dpi, bbox_inches='tight')
plt.show()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3))

for lang in languages:
    df = pd.read_csv(f'MAXN_SUPER/{lang}/ExponentialBackoff/5.5/telemetry_RGB.csv')
    time_seq = range(len(df))

    print(f"{lang}: maximum RSS: {df['rss_bytes'].max():,}, steady state: {df['rss_bytes'].iloc[10:].max():,}")
    rss_mb = df['rss_bytes'] / (1024 * 1024)
    ax1.plot(time_seq, rss_mb, color=lang_colours[lang], linewidth=2, label=lang_labels[lang])

    if lang in ['cpp', 'rust']:
        alloc_mb = df['allocated_bytes'] / (1024 * 1024)
        cumulative_alloc = alloc_mb.cumsum()
        ax2.plot(time_seq, cumulative_alloc, color=lang_colours[lang], linewidth=2, label=lang_labels[lang])
        print(f"{lang} dynamic allocation rate: {df['allocated_bytes'].mean():,} Bytes/second, steady state: {df['allocated_bytes'].iloc[10:].mean():,}")
        print(f"{lang} fordblks drift: {(df['fordblks_bytes'].max() - df['fordblks_bytes'].min()):,} Bytes, "
            f"steady state: {(df['fordblks_bytes'].iloc[10:].max() - df['fordblks_bytes'].iloc[10:].min()):,} Bytes")

    print("")

ax1.set_xlabel('Time (seconds)')
ax1.set_ylabel('Memory Usage (MiB)')
ax1.legend(loc="center right", frameon=True, facecolor='white', edgecolor='gray', framealpha=1.0)

ax2.set_xlabel('Time (seconds)')
ax2.set_ylabel('Total Allocated Memory (MiB)')
ax2.legend(loc="center right", frameon=True, facecolor='white', edgecolor='gray', framealpha=1.0)

fig.tight_layout()
plt.grid(linestyle='--', alpha=1.0)
plt.subplots_adjust(wspace=0.35)
plt.savefig('img/MAXN_SUPER-memory-profiling.pdf', bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
figsize = (10, 5)

plt.figure(figsize=figsize)
for lang in languages:
    df = pd.read_csv(f'MAXN_SUPER/{lang}/ExponentialBackoff/1.0/tegrastats.csv')
    plt.plot(range(len(df)), df['cpu_temp'], color=lang_colours[lang], linewidth=2, label=lang_labels[lang])
    print(f"{lang} 1.0 max temp time: {df['cpu_temp'].idxmax()}")
    df_tel = pd.read_csv(f'MAXN_SUPER/{lang}/ExponentialBackoff/1.0/telemetry_RGB.csv')
    triggered_rows = df_tel[df_tel['fan_pwm'] > 0]

    if not triggered_rows.empty:
        trigger_second = triggered_rows.index[0]
        print(f"{lang:<6}: fan triggered at {trigger_second} seconds")
    else:
        print(f"{lang:<6}: Fan never triggered (Max temp: {df['cpu_temp'].max()}°C)")

plt.axhline(y=74.0, color=linecol, linestyle=linestyle, linewidth=linewidth, label="Fan Trigger (74°C)")
plt.xlabel("Time (seconds)")
plt.ylabel("CPU Temperature (°C)")
plt.grid(linestyle='--', alpha=0.7)
plt.legend(loc='center right', frameon=True, facecolor='white', edgecolor='gray', framealpha=1.0)
plt.tight_layout()
plt.savefig(f'img/MAXN_SUPER-thermals-native.pdf', dpi=img_dpi)
plt.show()

plt.figure(figsize=figsize)
for lang in languages:
    df = pd.read_csv(f'MAXN_SUPER/{lang}/ExponentialBackoff/{'0.04' if lang == 'python' else '5.5'}/tegrastats.csv')
    plt.plot(range(len(df)), df['cpu_temp'], color=lang_colours[lang], linewidth=2, label=lang_labels[lang])
    print(f"{lang} 0.05 max temp time: {df['cpu_temp'].idxmax()}")

    df_tel = pd.read_csv(f'MAXN_SUPER/{lang}/ExponentialBackoff/{'0.04' if lang == 'python' else '5.5'}/telemetry_RGB.csv')
    triggered_rows = df_tel[df_tel['fan_pwm'] > 0]

    if not triggered_rows.empty:
        trigger_second = triggered_rows.index[0]
        print(f"{lang:<6}: fan triggered at {trigger_second} seconds")
    else:
        print(f"{lang:<6}: Fan never triggered (Max temp: {df['cpu_temp'].max()}°C)")

plt.axhline(y=74.0, color=linecol, linestyle=linestyle, linewidth=linewidth, label="Fan Trigger (74°C)")
plt.xlabel("Time (seconds)")
plt.ylabel("CPU Temperature (°C)")
plt.grid(linestyle='--', alpha=1.0)
plt.legend(loc='center right', frameon=True, facecolor='white', edgecolor='gray', framealpha=1.0)
plt.tight_layout()
plt.savefig(f'img/MAXN_SUPER-thermals-saturated.pdf', dpi=img_dpi)
plt.show()

In [ ]:
df_cpp = pd.read_csv('MAXN_SUPER/cpp/ExponentialBackoff/0.04/telemetry_RGB.csv').iloc[10:]
lat_cpp = df_cpp['total_latency_p50']

df_rust = pd.read_csv('MAXN_SUPER/rust/ExponentialBackoff/0.04/telemetry_RGB.csv').iloc[10:]
lat_rust = df_rust['total_latency_p50']

df_py = pd.read_csv('MAXN_SUPER/python/ExponentialBackoff/0.04/telemetry_RGB.csv').iloc[10:]
lat_py = df_py['total_latency_p50']

H, p_val = stats.kruskal(lat_cpp, lat_rust, lat_py)
n = len(lat_cpp) + len(lat_rust) + len(lat_py)
epsilon_sq = (H - 3 + 1) / (n - 3)
dunn = sp.posthoc_dunn([lat_cpp, lat_rust, lat_py], p_adjust='bonferroni')

print("**Kruskal-Wallis**")
print(f"H-statistic: {H:.2f}, p-value: {p_val}, Epsilon-squared: {epsilon_sq:.4f}")
print("Dunn's p-values (1=C++, 2=Rust, 3=Python):\n", dunn)


print("\n\n**Spearman's (GC/latency)**")
rho_gc, p_gc = stats.spearmanr(df_py['gc_pause_ns'], df_py['total_latency_max'])
print(f"rho = {rho_gc:.4f} (p = {p_gc})")


df_cpp_55 = pd.read_csv('MAXN_SUPER/cpp/ExponentialBackoff/5.5/telemetry_RGB.csv').iloc[10:]
df_teg_55 = pd.read_csv('MAXN_SUPER/cpp/ExponentialBackoff/5.5/tegrastats.csv').iloc[10:]

df_cpp_55['timestamp_sec'] = (df_cpp_55['timestamp_ns'] // 1_000_000_000).astype('int64')
df_cpp_55 = df_cpp_55.sort_values('timestamp_sec')

df_teg_55['timestamp'] = df_teg_55['timestamp'].astype('int64')
df_teg_55 = df_teg_55.sort_values('timestamp')

df_merged_55 = pd.merge_asof(df_cpp_55, df_teg_55, left_on='timestamp_sec', right_on='timestamp', direction='nearest')
rho_temp, p_temp = stats.spearmanr(df_merged_55['allocated_bytes'], df_merged_55['cpu_temp'])

print("\n\n**Spearman's (C++ Allocation/Temp)**")
print(f"C++ Allocated Bytes vs CPU Temp (Load 5.5): rho = {rho_temp:.4f} (p = {p_temp})")


df_rust_55 = pd.read_csv('MAXN_SUPER/rust/ExponentialBackoff/5.5/telemetry_RGB.csv').iloc[10:]
df_teg_55 = pd.read_csv('MAXN_SUPER/rust/ExponentialBackoff/5.5/tegrastats.csv').iloc[10:]

df_rust_55['timestamp_sec'] = (df_rust_55['timestamp_ns'] // 1_000_000_000).astype('int64')
df_rust_55 = df_rust_55.sort_values('timestamp_sec')

df_teg_55['timestamp'] = df_teg_55['timestamp'].astype('int64')
df_teg_55 = df_teg_55.sort_values('timestamp')

df_merged_55 = pd.merge_asof(df_rust_55, df_teg_55, left_on='timestamp_sec', right_on='timestamp', direction='nearest')
rho_temp, p_temp = stats.spearmanr(df_merged_55['allocated_bytes'], df_merged_55['cpu_temp'])

print("\n\n**Spearman's (Rust Allocation/Temp)**")
print(f"Rust Allocated Bytes vs CPU Temp (Load 5.5): rho = {rho_temp:.4f} (p = {p_temp})")

df_cpp_7w = pd.read_csv('07-watt/cpp/ExponentialBackoff/5.5/telemetry_RGB.csv').iloc[10:]
df_teg_cpp_7w = pd.read_csv('07-watt/cpp/ExponentialBackoff/5.5/tegrastats.csv').iloc[10:]

df_cpp_7w['timestamp_sec'] = (df_cpp_7w['timestamp_ns'] // 1_000_000_000).astype('int64')
df_cpp_7w = df_cpp_7w.sort_values('timestamp_sec')

df_teg_cpp_7w['timestamp'] = df_teg_cpp_7w['timestamp'].astype('int64')
df_teg_cpp_7w = df_teg_cpp_7w.sort_values('timestamp')

df_merged_cpp_7w = pd.merge_asof(df_cpp_7w, df_teg_cpp_7w, left_on='timestamp_sec', right_on='timestamp', direction='nearest')

rho_dvfs_cpp, p_dvfs_cpp = stats.spearmanr(df_merged_cpp_7w['cpu_temp'], df_merged_cpp_7w['inference_exec_p50'])

print("\n\n**Spearman's (7W Mode)**")
print(f"C++ CPU Temp vs Inference Latency (7W Load 5.5): rho = {rho_dvfs_cpp:.4f} (p = {p_dvfs_cpp})")

df_rust_7w = pd.read_csv('07-watt/rust/ExponentialBackoff/5.5/telemetry_RGB.csv').iloc[10:]
df_teg_7w = pd.read_csv('07-watt/rust/ExponentialBackoff/5.5/tegrastats.csv').iloc[10:]

df_rust_7w['timestamp_sec'] = (df_rust_7w['timestamp_ns'] // 1_000_000_000).astype('int64')
df_rust_7w = df_rust_7w.sort_values('timestamp_sec')

df_teg_7w['timestamp'] = df_teg_7w['timestamp'].astype('int64')
df_teg_7w = df_teg_7w.sort_values('timestamp')

df_merged_7w = pd.merge_asof(df_rust_7w, df_teg_7w, left_on='timestamp_sec', right_on='timestamp', direction='nearest')

rho_dvfs, p_dvfs = stats.spearmanr(df_merged_7w['cpu_temp'], df_merged_7w['inference_exec_p50'])

print(f"Rust CPU Temp vs Inference Latency (7W Load 5.5): rho = {rho_dvfs:.4f} (p = {p_dvfs})")